In [18]:
!pip install torchvision
!pip install graphviz

In [13]:
#generate_dataset.py
import json
import random
NUM_SAMPLES=500
systems = [
"url shortener",
"chat application",
"video streaming platform",
"ride sharing service",
"ecommerce platform",
"social media platform",
"email service",
"file storage system"
]

components = {
"url shortener": ["User","API Gateway","URL Service","Cache","Database"],
"chat application": ["User","Gateway","Chat Service","Message Queue","Database"],
"video streaming platform": ["User","Gateway","Video Service","CDN","Storage"],
"ride sharing service": ["User","Gateway","Ride Service","Location Service","Database"],
"ecommerce platform": ["User","Gateway","Order Service","Payment Service","Database"],
"social media platform": ["User","Gateway","Feed Service","Media Service","Database"],
"email service": ["User","Gateway","Mail Service","Queue","Database"],
"file storage system": ["User","Gateway","Storage Service","Metadata Service","Database"]
}

data=[]

for i in range(NUM_SAMPLES):

    system=random.choice(systems)

    nodes=components[system]

    edges=[]

    for j in range(len(nodes)-1):
        edges.append([nodes[j],nodes[j+1]])

    data.append({
        "input":f"Design architecture for {system}",
        "nodes":nodes,
        "edges":edges
    })

with open("input.json","w") as f:
    json.dump(data,f,indent=2)

print("Created dataset with",len(data),"examples")

Created dataset with 500 examples


In [11]:
#model.py
import torch
import torch.nn as nn
import torch.nn.functional as F


class ResidualBlock(nn.Module):

    def __init__(self,dim):
        super().__init__()

        self.fc1=nn.Linear(dim,dim)
        self.fc2=nn.Linear(dim,dim)

    def forward(self,x):

        identity=x

        out=F.relu(self.fc1(x))
        out=self.fc2(out)

        out=out+identity

        return F.relu(out)


class CNNEncoder(nn.Module):

    def __init__(self,vocab_size,embed_dim):

        super().__init__()

        self.embedding=nn.Embedding(vocab_size,embed_dim)

        self.conv=nn.Conv1d(embed_dim,embed_dim,3,padding=1)

    def forward(self,x):

        x=self.embedding(x)

        x=x.permute(0,2,1)

        x=self.conv(x)

        x=x.permute(0,2,1)

        return x


class TransformerBlock(nn.Module):

    def __init__(self,embed_dim):

        super().__init__()

        self.attn=nn.MultiheadAttention(embed_dim,4,batch_first=True)

        self.norm=nn.LayerNorm(embed_dim)

    def forward(self,x):

        attn_out,_=self.attn(x,x,x)

        return self.norm(x+attn_out)


class ArchitectureModel(nn.Module):

    def __init__(self,vocab_size,embed_dim=128):

        super().__init__()

        self.encoder=CNNEncoder(vocab_size,embed_dim)

        self.transformer=TransformerBlock(embed_dim)

        self.residual=ResidualBlock(embed_dim)

        self.fc_nodes=nn.Linear(embed_dim,1)

        self.fc_edges=nn.Linear(embed_dim,1)

    def forward(self,x):

        x=self.encoder(x)

        x=self.transformer(x)

        x=self.residual(x)

        x=x.mean(dim=1)

        nodes=self.fc_nodes(x)

        edges=self.fc_edges(x)

        return nodes,edges

In [15]:
#train.py
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from torch.nn.utils.rnn import pad_sequence

#from model import ArchitectureModel

device="cuda" if torch.cuda.is_available() else "cpu"

with open("input.json") as f:
    data=json.load(f)

texts=[d["input"] for d in data]

vocab=set()

for t in texts:
    vocab.update(t.split())

vocab=list(vocab)

word2idx={w:i+1 for i,w in enumerate(vocab)}

with open("vocab.json","w") as f:
    json.dump(word2idx,f)

def encode(text):
    return torch.tensor([word2idx[w] for w in text.split()])


class ArchDataset(Dataset):

    def __init__(self,data):
        self.data=data

    def __len__(self):
        return len(self.data)

    def __getitem__(self,idx):

        item=self.data[idx]

        x=encode(item["input"])

        y_nodes=len(item["nodes"])
        y_edges=len(item["edges"])

        return x,torch.tensor(y_nodes),torch.tensor(y_edges)


dataset=ArchDataset(data)


def collate(batch):

    xs=[b[0] for b in batch]
    ys_nodes=[b[1] for b in batch]
    ys_edges=[b[2] for b in batch]

    xs=pad_sequence(xs,batch_first=True)

    ys_nodes=torch.stack(ys_nodes).float()
    ys_edges=torch.stack(ys_edges).float()

    return xs,ys_nodes,ys_edges


loader=DataLoader(dataset,batch_size=16,shuffle=True,collate_fn=collate)

model=ArchitectureModel(len(vocab)+1).to(device)

criterion=nn.MSELoss()

optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

for epoch in range(10):

    total_loss=0

    for xs,yn,ye in loader:

        xs=xs.to(device)
        yn=yn.to(device)
        ye=ye.to(device)

        optimizer.zero_grad()

        pred_nodes,pred_edges=model(xs)

        loss=criterion(pred_nodes.squeeze(),yn)+criterion(pred_edges.squeeze(),ye)

        loss.backward()

        optimizer.step()

        total_loss+=loss.item()

    print("Epoch",epoch,"Loss",total_loss)

torch.save(model.state_dict(),"model.pth")

print("Training complete")

Epoch 0 Loss 132.38770088367164
Epoch 1 Loss 0.7959527043858543
Epoch 2 Loss 0.047165869240416214
Epoch 3 Loss 0.009264873355277814
Epoch 4 Loss 0.0036395964307303075
Epoch 5 Loss 0.0016064737537817564
Epoch 6 Loss 0.0006934327120688977
Epoch 7 Loss 0.0003039690359400993
Epoch 8 Loss 0.0001255436739029392
Epoch 9 Loss 4.970095073986158e-05
Training complete


In [17]:
#test.py
import torch
import json
from graphviz import Digraph

#from model import ArchitectureModel

device="cuda" if torch.cuda.is_available() else "cpu"

with open("vocab.json") as f:
    word2idx=json.load(f)

model=ArchitectureModel(len(word2idx)+1)

model.load_state_dict(torch.load("model.pth"))

model.to(device)
model.eval()


def encode(text):

    words=text.split()

    return torch.tensor([[word2idx.get(w,0) for w in words]])


def generate_diagram(nodes,edges):

    dot=Digraph()

    for n in nodes:
        dot.node(n)

    for s,d in edges:
        dot.edge(s,d)

    dot.render("architecture_output",format="png")

    print("Saved architecture_output.png")


text="Design architecture for chat application"

encoded=encode(text).to(device)

nodes_pred,edges_pred=model(encoded)

print("Predicted nodes:",nodes_pred.item())
print("Predicted edges:",edges_pred.item())


nodes=["User","Gateway","Chat Service","Queue","Database"]

edges=[
("User","Gateway"),
("Gateway","Chat Service"),
("Chat Service","Queue"),
("Queue","Database")
]

generate_diagram(nodes,edges)

Predicted nodes: 5.00346565246582
Predicted edges: 4.000821590423584
Saved architecture_output.png
